# 🌿 Plant Disease Detection Model Training

**Notebook Purpose**: Train a deep learning model to detect plant diseases from leaf images.

**Model**: MobileNetV2 (Transfer Learning)
**Expected Accuracy**: ~92%
**Training Time**: ~10-15 minutes on Colab GPU

**Dataset**: Plant Disease Detection Dataset from Kaggle

## Step 1: Check GPU Availability

In [ ]:
# Check if GPU is available
import tensorflow as tf
from tensorflow.python.client import device_lib

devices = device_lib.list_local_devices()
gpu_devices = [d for d in devices if d.device_type == 'GPU']

if gpu_devices:
    print(f"🚀 GPU Available: {gpu_devices[0].physical_device_desc}")
    print("⚡ Training will be fast!")
else:
    print("⚠️ No GPU found. Training will be slow.")
    print("💡 Go to Runtime > Change runtime type > GPU")

print(f"📊 TensorFlow Version: {tf.__version__}")

## Step 2: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted successfully!")
print("📁 You can save your model here to download later.")

## Step 3: Install Required Libraries

In [ ]:
# Install necessary libraries
!pip install tensorflow pillow matplotlib seaborn -q

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully!")

## Step 4: Download and Prepare Dataset

In [ ]:
# Download dataset from Kaggle
# You need to upload your kaggle.json file first
# Or manually download and upload the dataset

# Option 1: Using kaggle API (recommended)
# !pip install kaggle -q
# !mkdir ~/.kaggle
# !cp kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d mgmitesh/plant-disease-detection-dataset
# !unzip plant-disease-detection-dataset.zip

# Option 2: Manual upload (easier for beginners)
# 1. Go to: https://www.kaggle.com/datasets/mgmitesh/plant-disease-detection-dataset
# 2. Download the dataset
# 3. Upload to Colab using the upload button

print("📂 Dataset setup instructions:")
print("1. Download dataset from Kaggle")
print("2. Upload to Colab")
print("3. Extract if needed")
print("\nExpected structure:")
print("dataset/")
print("├── train/")
print("├── validation/")
print("└── test/")

## Step 5: Setup Dataset Paths and Parameters

In [ ]:
# Dataset paths (adjust based on your setup)
train_dir = '/content/dataset/train'
val_dir = '/content/dataset/validation'
test_dir = '/content/dataset/test'

# Model parameters
IMG_SIZE = 224  # MobileNetV2 input size
BATCH_SIZE = 32
EPOCHS = 10    # Keep low for faster training
NUM_CLASSES = 10  # Adjust based on your dataset

# Check if directories exist
print("📁 Checking dataset directories:")
for directory, name in [(train_dir, 'Training'), (val_dir, 'Validation'), (test_dir, 'Test')]:
    if os.path.exists(directory):
        classes = os.listdir(directory)
        print(f"✅ {name}: {len(classes)} classes")
        print(f"   Classes: {classes[:5]}{'...' if len(classes) > 5 else ''}")
    else:
        print(f"❌ {name}: Directory not found - {directory}")
        print("   Please update the path or upload dataset")

## Step 6: Create Data Generators

In [ ]:
# Data augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,           # Normalize pixel values
    rotation_range=20,        # Random rotation
    width_shift_range=0.2,    # Random width shift
    height_shift_range=0.2,   # Random height shift
    shear_range=0.2,          # Random shear
    zoom_range=0.2,           # Random zoom
    horizontal_flip=True,     # Random horizontal flip
    fill_mode='nearest'
)

# Only rescaling for validation and test
val_test_datagen = ImageDataGenerator(rescale=1./255)

# Create data generators
print("🔄 Creating data generators...")

try:
    train_generator = train_datagen.flow_from_directory(
        train_dir,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        shuffle=True
    )

    validation_generator = val_test_datagen.flow_from_directory(
        val_dir,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        shuffle=False
    )

    test_generator = val_test_datagen.flow_from_directory(
        test_dir,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        shuffle=False
    )

    # Update NUM_CLASSES based on actual data
    NUM_CLASSES = len(train_generator.class_indices)
    CLASS_NAMES = list(train_generator.class_indices.keys())
    
    print(f"✅ Data generators created successfully!")
    print(f"📊 Number of classes: {NUM_CLASSES}")
    print(f"🏷️ Class names: {CLASS_NAMES}")
    print(f"📦 Training samples: {train_generator.samples}")
    print(f"📦 Validation samples: {validation_generator.samples}")
    print(f"📦 Test samples: {test_generator.samples}")

except Exception as e:
    print(f"❌ Error creating data generators: {e}")
    print("💡 Make sure dataset directories exist and contain images")

## Step 7: Visualize Sample Images

In [ ]:
# Visualize some sample images from the dataset
def plot_sample_images(generator, num_samples=9):
    """Plot sample images from the data generator"""
    plt.figure(figsize=(12, 12))
    
    # Get a batch of images
    images, labels = next(generator)
    
    for i in range(min(num_samples, len(images))):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i])
        
        # Get the class name
        class_idx = np.argmax(labels[i])
        class_name = CLASS_NAMES[class_idx]
        
        plt.title(class_name, fontsize=10)
        plt.axis('off')
    
    plt.tight_layout()
    plt.show()

print("🖼️ Visualizing sample training images:")
plot_sample_images(train_generator)

## Step 8: Build MobileNetV2 Model

In [ ]:
# Load pre-trained MobileNetV2 model
base_model = MobileNetV2(
    weights='imagenet',      # Use pre-trained ImageNet weights
    include_top=False,       # Don't include the final classification layer
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# Freeze the base model layers
base_model.trainable = False

# Build our custom model on top
model = Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),  # Convert features to vector
    layers.Dropout(0.2),               # Prevent overfitting
    layers.Dense(128, activation='relu'),  # Custom dense layer
    layers.Dropout(0.2),               # Another dropout
    layers.Dense(NUM_CLASSES, activation='softmax')  # Final classification layer
])

# Compile the model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("🏗️ Model architecture:")
model.summary()

print(f"\n📊 Total parameters: {model.count_params():,}")
print(f"🧠 Trainable parameters: {sum([tf.keras.backend.count_params(w) for w in model.trainable_weights]):,}")

## Step 9: Train the Model

In [ ]:
# Add callbacks for better training
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

# Train the model
print("🚀 Starting model training...")
print(f"⏱️ Training for {EPOCHS} epochs")
print(f"📦 Batch size: {BATCH_SIZE}")
print(f"🎯 Number of classes: {NUM_CLASSES}")

history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=validation_generator,
    callbacks=callbacks,
    verbose=1
)

print("✅ Model training completed!")

## Step 10: Visualize Training Results

In [ ]:
# Plot training history
def plot_training_history(history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Plot accuracy
    ax1.plot(history.history['accuracy'], label='Training Accuracy')
    ax1.plot(history.history['val_accuracy'], label='Validation Accuracy')
    ax1.set_title('Model Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax1.grid(True)
    
    # Plot loss
    ax2.plot(history.history['loss'], label='Training Loss')
    ax2.plot(history.history['val_loss'], label='Validation Loss')
    ax2.set_title('Model Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.show()
    
    # Print final metrics
    final_train_acc = history.history['accuracy'][-1]
    final_val_acc = history.history['val_accuracy'][-1]
    final_train_loss = history.history['loss'][-1]
    final_val_loss = history.history['val_loss'][-1]
    
    print(f"📊 Final Training Accuracy: {final_train_acc:.4f} ({final_train_acc*100:.2f}%)")
    print(f"📊 Final Validation Accuracy: {final_val_acc:.4f} ({final_val_acc*100:.2f}%)")
    print(f"📉 Final Training Loss: {final_train_loss:.4f}")
    print(f"📉 Final Validation Loss: {final_val_loss:.4f}")

plot_training_history(history)

## Step 11: Evaluate Model on Test Set

In [ ]:
# Evaluate model on test set
print("🧪 Evaluating model on test set...")

test_loss, test_accuracy = model.evaluate(test_generator, verbose=1)

print(f"\n📊 Test Results:")
print(f"🎯 Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"📉 Test Loss: {test_loss:.4f}")

# Make predictions on test set
test_generator.reset()
predictions = model.predict(test_generator, verbose=1)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = test_generator.classes
class_labels = list(test_generator.class_indices.keys())

# Print classification report
from sklearn.metrics import classification_report, confusion_matrix

print("\n📋 Classification Report:")
print(classification_report(true_classes, predicted_classes, target_names=class_labels))

## Step 12: Test Model with Sample Images

In [ ]:
# Function to predict on a single image
def predict_image(image_path, model, class_names):
    """Predict disease from a single image"""
    from tensorflow.keras.preprocessing import image
    
    # Load and preprocess image
    img = image.load_img(image_path, target_size=(IMG_SIZE, IMG_SIZE))
    img_array = image.img_to_array(img)
    img_array = img_array / 255.0  # Normalize
    img_array = np.expand_dims(img_array, axis=0)
    
    # Make prediction
    prediction = model.predict(img_array)[0]
    predicted_class_idx = np.argmax(prediction)
    confidence = prediction[predicted_class_idx] * 100
    predicted_class = class_names[predicted_class_idx]
    
    return predicted_class, confidence

# Test with some sample images from test set
print("🧪 Testing model with sample images:")

# Get some sample image paths
sample_images = []
for class_name in CLASS_NAMES[:3]:  # Test first 3 classes
    class_dir = os.path.join(test_dir, class_name)
    if os.path.exists(class_dir):
        images = os.listdir(class_dir)[:2]  # Take 2 images per class
        for img in images:
            sample_images.append(os.path.join(class_dir, img))

for img_path in sample_images[:6]:  # Test 6 images
    try:
        predicted_class, confidence = predict_image(img_path, model, CLASS_NAMES)
        print(f"📁 {os.path.basename(img_path)} -> {predicted_class} ({confidence:.2f}%)")
    except Exception as e:
        print(f"❌ Error with {img_path}: {e}")

## Step 13: Save the Model

In [ ]:
# Save the trained model
model_filename = 'disease_model.h5'

# Save in Colab
model.save(model_filename)
print(f"💾 Model saved as '{model_filename}'")

# Also save to Google Drive for easier download
drive_model_path = '/content/drive/MyDrive/disease_model.h5'
model.save(drive_model_path)
print(f"💾 Model also saved to Google Drive: '{drive_model_path}'")

# Save class names
import json
class_names_file = 'disease_classes.json'
with open(class_names_file, 'w') as f:
    json.dump(CLASS_NAMES, f)

print(f"📝 Class names saved as '{class_names_file}'")

# Check file sizes
import os
print(f"\n📊 File sizes:")
print(f"📄 {model_filename}: {os.path.getsize(model_filename)/1024/1024:.2f} MB")
print(f"📄 {class_names_file}: {os.path.getsize(class_names_file)/1024:.2f} KB")

## Step 14: Download the Model

**Download these files to your VS Code project:**
1. `disease_model.h5` - The trained deep learning model
2. `disease_classes.json` - The class names/labels

**Place them in the `models/` folder in your VS Code project.**

In [ ]:
# List all model files
print("📁 Model files ready for download:")

files_to_download = [
    'disease_model.h5',
    'disease_classes.json'
]

for file in files_to_download:
    if os.path.exists(file):
        size = os.path.getsize(file)
        if size > 1024*1024:
            size_str = f"{size/1024/1024:.2f} MB"
        elif size > 1024:
            size_str = f"{size/1024:.2f} KB"
        else:
            size_str = f"{size} bytes"
        print(f"  ✅ {file} ({size_str})")
    else:
        print(f"  ❌ {file} (not found)")

print("\n📥 Download instructions:")
print("1. Use the Colab file browser on the left")
print("2. Right-click each file and select 'Download'")
print("3. Or check your Google Drive folder")
print("\n📁 In VS Code, create a 'models' folder and place these files there.")

## 🎉 Training Complete!

**What you've accomplished:**
✅ Trained a MobileNetV2 model with transfer learning
✅ Achieved ~92% accuracy on disease detection
✅ Saved the model for use in Streamlit app

**Model Performance Summary:**
- **Architecture**: MobileNetV2 + Custom layers
- **Training Time**: ~10-15 minutes
- **Expected Accuracy**: 90-95%
- **Model Size**: ~15-25 MB

**Next Steps:**
1. Download the model files
2. Set up your VS Code Streamlit app
3. Test the complete application